# 08 · Accelerate — M×N GPU (subprocess.Popen)

`accelerate launch` CLI를 `subprocess.Popen`으로 driver에서 띄웁니다. 노트북 매직(`!`, `%sh`)은 PATH/auth 문제로 노트북 스코프 env와 충돌하기 쉬워, 명령 문자열을 `shlex.split` → `Popen`으로 직접 실행하고 자식 프로세스에 `DATABRICKS_HOST`/`DATABRICKS_TOKEN`을 명시 주입합니다.

`--num_processes`는 지정하지 않습니다. Accelerate가 가시 GPU 수만큼 자동으로 띄웁니다. 토폴로지 1×1 / 1×N / M×N은 클러스터 사양으로만 결정되며, 이 노트북 한 개로 전부 커버합니다.

Entry point는 [`torch_distributor_trainer.py`](https://github.com/Aiden-Jeon/databricks-distributed-training/blob/main/02-script-based/torch_distributor_trainer.py)입니다. TorchDistributor / Accelerate가 공통으로 사용하도록 설계되었습니다(`if __name__ == '__main__':` + argparse).

**클러스터**: Classic GPU, DBR 17.3 LTS ML. 단일 fat-node (예: `g5.12xlarge` 4×A10) 또는 multi-node 환경에서 그대로 동작합니다.

In [ ]:
%run ./00-setup

## Entry point 스크립트 경로

노트북과 같은 워크스페이스 폴더의 `torch_distributor_trainer.py`를 `accelerate launch` 대상으로 넘깁니다.

In [ ]:
import os

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
SCRIPT_PATH = os.path.join(SCRIPT_DIR, "torch_distributor_trainer.py")
print(f"SCRIPT_PATH={SCRIPT_PATH}")

## MLflow run 생성

driver에서 run을 시작하고 `run_id`만 빼서 자식 프로세스에 넘깁니다. rank 0 자식이 같은 run에 attach해 system metrics·per-epoch 메트릭을 누적합니다.

In [ ]:
import mlflow

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="acc-MxN", log_system_metrics=True) as run:
    RUN_ID = run.info.run_id
print(f"RUN_ID={RUN_ID}")

## `accelerate launch` 명령 + subprocess.Popen

이 셀에서 다음 다섯 가지를 처리합니다.

1. `inference_cmd`를 f-string으로 조립하고 `shlex.split`으로 토큰화합니다.
2. `sub_env`에 `DATABRICKS_HOST` / `DATABRICKS_TOKEN`을 주입합니다. 자식 프로세스는 노트북 auth context를 상속하지 않으므로, MLflow / WorkspaceClient 호출을 위해 명시 전달이 필요합니다.
3. Py4J keepalive 스레드를 둡니다. 장시간 학습 중 driver Py4J gateway가 끊겨 후처리가 폭사하는 것을 막기 위해 5분마다 `SELECT 1`을 찌릅니다.
4. `stderr=STDOUT` + `bufsize=1` + `text=True`로 stdout/stderr를 한 스트림 라인 단위로 출력합니다.
5. non-zero return 시 `dbutils.notebook.exit`로 실패 종료시켜 DAB job retry가 동작하게 합니다.

In [ ]:
import shlex
import subprocess
import threading

tower_hidden_args = " ".join(str(x) for x in cfg["tower_hidden"])
ckpt_path = os.path.join(CKPT_DIR, "acc-MxN.pt")

inference_cmd = f"""accelerate launch --mixed_precision=bf16 {SCRIPT_PATH} \
    --run_id '{RUN_ID}' \
    --db_host '{DB_HOST}' \
    --db_token '{DB_TOKEN}' \
    --data_dir '{DATA_DIR}' \
    --ckpt_path '{ckpt_path}' \
    --n_users {cfg['n_users']} \
    --n_items {cfg['n_items']} \
    --emb_dim {cfg['emb_dim']} \
    --tower_hidden {tower_hidden_args} \
    --batch_size {cfg['batch_size_per_gpu']} \
    --num_epochs {cfg['num_epochs']} \
    --max_steps_per_epoch {cfg['max_steps_per_epoch']} \
    --patience {PATIENCE} \
    --min_delta {MIN_DELTA} \
    --topology MxN \
    --script_dir '{SCRIPT_DIR}'
"""

sub_env = os.environ.copy()
sub_env["DATABRICKS_HOST"] = DB_HOST
sub_env["DATABRICKS_TOKEN"] = DB_TOKEN

done_event = threading.Event()

def _keepalive(interval=300):
    while not done_event.is_set():
        try:
            spark.sql("SELECT 1").collect()
        except Exception:
            pass
        done_event.wait(interval)

keepalive_thread = threading.Thread(target=_keepalive, daemon=True)
keepalive_thread.start()

p = subprocess.Popen(
    shlex.split(inference_cmd),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=sub_env,
)

for line in p.stdout:
    print(line, end="")

p.wait()
done_event.set()

if p.returncode != 0:
    dbutils.notebook.exit(f"accelerate launch failed: rc={p.returncode}")